In [138]:
"""
eval_proof_lora.py
════════════════════════════════════════════════════════════════════
Side-by-side eval: base gemma-2-9b-it vs fine-tuned LoRA adapter
Renders a clean HTML report directly in Colab.
Markdown in model outputs is rendered properly.
Author: Kunj Shah (kunjcr2)
════════════════════════════════════════════════════════════════════
"""

import re
import torch
from IPython.display import display, HTML
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
from peft import PeftModel

# ════════════════════════════════════════════════════════════════
# CONFIG
# ════════════════════════════════════════════════════════════════

BASE_MODEL_ID = "google/gemma-2-9b-it"
LORA_REPO_ID  = "kunjcr2/gemma-2-9b-proof-lora"
DEVICE        = "cuda"

GEN_CONFIG = GenerationConfig(
    max_new_tokens=512,
    temperature=0.3,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.2,
)

SYSTEM = (
    "You are a formal mathematics assistant. "
    "When given a mathematical statement or theorem, produce a rigorous, "
    "step-by-step proof, and every step in a new line. Structure your proof with clearly labelled steps: "
    "Given, Definitions, Lemma (if needed), Proof, and QED."
)

EVAL_STATEMENTS = [
    # "Prove pythagorean theoream for me step by step."
    # "The square root of 2 is irrational.",
    # "There are infinitely many prime numbers.",
    "For any integer n, n² + n is even.",
    # "If p is prime and p divides ab, then p divides a or p divides b.",
    # "Prove that the set of rational numbers is countable",
    # "The product of two odd integers is odd.",
    # "There is no largest prime number.",
    # "The sum of angles in any triangle is 180°.",
]

In [139]:
# ════════════════════════════════════════════════════════════════
# LOAD MODELS
# ════════════════════════════════════════════════════════════════

def load_models():
    print("[LOAD] Tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print("[LOAD] Base model...")
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        attn_implementation="eager",
    )
    base_model.eval()

    print("[LOAD] LoRA adapter...")
    lora_model = PeftModel.from_pretrained(base_model, LORA_REPO_ID)
    lora_model.eval()

    print("[LOAD] Done.")
    return tokenizer, base_model, lora_model

In [140]:
# ════════════════════════════════════════════════════════════════
# INFERENCE
# ════════════════════════════════════════════════════════════════

def build_prompt(statement: str) -> str:
    return (
        f"<bos><start_of_turn>user\n"
        f"{SYSTEM}\n\nProve the following:\n\n{statement}"
        f"<end_of_turn>\n<start_of_turn>model\n"
    )

def generate(model, tokenizer, statement: str) -> str:
    prompt = build_prompt(statement)
    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=512
    ).to(DEVICE)
    with torch.no_grad():
        out = model.generate(**inputs, generation_config=GEN_CONFIG)
    response = tokenizer.decode(
        out[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True
    )
    return response.strip()

In [141]:
# ════════════════════════════════════════════════════════════════
# MARKDOWN → HTML
# ════════════════════════════════════════════════════════════════

def markdown_to_html(text: str) -> str:
    text = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    text = re.sub(r'\*\*(.+?)\*\*', r'<strong>\1</strong>', text)
    text = re.sub(r'(?<!\*)\*(?!\*)(.+?)(?<!\*)\*(?!\*)', r'<em>\1</em>', text)
    text = re.sub(r'`(.+?)`', r'<code>\1</code>', text)
    text = re.sub(r'^### (.+)$', r'<h4>\1</h4>', text, flags=re.MULTILINE)
    text = re.sub(r'^## (.+)$',  r'<h3>\1</h3>', text, flags=re.MULTILINE)
    text = re.sub(r'^# (.+)$',   r'<h2>\1</h2>', text, flags=re.MULTILINE)
    # newlines but not inside LaTeX blocks
    parts = re.split(r'(\$\$?.+?\$\$?)', text, flags=re.DOTALL)
    text = "".join(p if p.startswith('$') else p.replace("\n", "<br>") for p in parts)
    return text

In [142]:
from IPython.display import Javascript
display(Javascript('MathJax.typesetPromise()'))

<IPython.core.display.Javascript object>

In [143]:
# ════════════════════════════════════════════════════════════════
# HTML RENDERER
# ════════════════════════════════════════════════════════════════

def render_html(results: list):
    cards = ""
    for i, r in enumerate(results):
        cards += f"""
        <div class="card">
            <div class="statement">
                <span class="idx">#{i+1}</span>
                {markdown_to_html(r['statement'])}
            </div>
            <div class="cols">
                <div class="col base">
                    <div class="col-header">
                        <span class="dot dot-red"></span>
                        Base Model
                    </div>
                    <div class="proof-text">{markdown_to_html(r['base'])}</div>
                </div>
                <div class="col lora">
                    <div class="col-header">
                        <span class="dot dot-green"></span>
                        Fine-tuned LoRA
                    </div>
                    <div class="proof-text">{markdown_to_html(r['lora'])}</div>
                </div>
            </div>
        </div>
        """

    html = f"""
        <script>
        window.MathJax = {{
            tex: {{
                inlineMath: [['$', '$'], ['\\\\(', '\\\\)']],
                displayMath: [['$$', '$$'], ['\\\\[', '\\\\]']],
                packages: {{'[+]': ['ams']}},
                processEscapes: true
            }},
            options: {{
                skipHtmlTags: ['script', 'noscript', 'style', 'textarea', 'pre']
            }},
            startup: {{
                ready() {{
                    MathJax.startup.defaultReady();
                }}
            }},
            svg: {{ fontCache: 'global' }}
        }};
        </script>
        <script src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-chtml-full.js"></script>    <style>
        @import url('https://fonts.googleapis.com/css2?family=IBM+Plex+Mono:wght@400;600&family=Fraunces:ital,wght@0,300;0,600;1,300&display=swap');

        .eval-wrap {{
            font-family: 'Fraunces', Georgia, serif;
            background: #0d0d0d;
            color: #e8e2d9;
            padding: 32px 24px;
            border-radius: 12px;
            max-width: 1200px;
            margin: 0 auto;
        }}

        .eval-title {{
            font-size: 28px;
            font-weight: 600;
            letter-spacing: -0.5px;
            margin-bottom: 4px;
            color: #f5f0e8;
        }}

        .eval-sub {{
            font-size: 13px;
            font-family: 'IBM Plex Mono', monospace;
            color: #666;
            margin-bottom: 32px;
        }}

        .card {{
            background: #141414;
            border: 1px solid #242424;
            border-radius: 10px;
            margin-bottom: 24px;
            overflow: hidden;
        }}

        .statement {{
            padding: 16px 20px;
            background: #1a1a1a;
            border-bottom: 1px solid #242424;
            font-size: 15px;
            font-style: italic;
            color: #c8bfb0;
            line-height: 1.5;
        }}

        .idx {{
            font-family: 'IBM Plex Mono', monospace;
            font-size: 11px;
            color: #555;
            margin-right: 10px;
            font-style: normal;
        }}

        .cols {{
            display: grid;
            grid-template-columns: 1fr 1fr;
        }}

        .col {{ padding: 20px; }}

        .col.base {{ border-right: 1px solid #242424; }}

        .col-header {{
            font-family: 'IBM Plex Mono', monospace;
            font-size: 11px;
            font-weight: 600;
            letter-spacing: 1.5px;
            text-transform: uppercase;
            margin-bottom: 14px;
            display: flex;
            align-items: center;
            gap: 8px;
        }}

        .dot {{
            width: 7px;
            height: 7px;
            border-radius: 50%;
            display: inline-block;
        }}

        .dot-red   {{ background: #c0392b; }}
        .dot-green {{ background: #27ae60; }}

        .col.base .col-header {{ color: #c0392b; }}
        .col.lora .col-header {{ color: #27ae60; }}

        .proof-text {{
            font-family: 'IBM Plex Mono', monospace;
            font-size: 12px;
            line-height: 1.9;
            color: #a09888;
        }}

        .proof-text strong {{ color: #d4c9b8; font-weight: 600; }}
        .proof-text em     {{ color: #b8a898; font-style: italic; }}
        .proof-text code   {{
            background: #222;
            padding: 1px 5px;
            border-radius: 3px;
            font-size: 11px;
            color: #e0d5c5;
        }}
        .proof-text h2, .proof-text h3, .proof-text h4 {{
            font-family: 'Fraunces', serif;
            color: #d4c9b8;
            margin: 8px 0 4px 0;
        }}

        .col.lora .proof-text          {{ color: #b8d4b0; }}
        .col.lora .proof-text strong   {{ color: #d4ecd0; }}
        .col.lora .proof-text em       {{ color: #a8c8a0; }}
        .col.lora .proof-text code     {{ background: #1a2a1a; color: #c8e8c0; }}
        .col.lora .proof-text h2,
        .col.lora .proof-text h3,
        .col.lora .proof-text h4       {{ color: #c8e0c0; }}

        .score-bar {{
            display: flex;
            gap: 16px;
            margin-bottom: 28px;
            flex-wrap: wrap;
        }}

        .score-chip {{
            font-family: 'IBM Plex Mono', monospace;
            font-size: 11px;
            padding: 6px 12px;
            border-radius: 20px;
            background: #1a1a1a;
            border: 1px solid #2a2a2a;
            color: #888;
        }}

        .score-chip span {{
            color: #27ae60;
            font-weight: 600;
        }}
    </style>

    <div class="eval-wrap">
        <div class="eval-title">Proof Writing — Before vs After</div>
        <div class="eval-sub">gemma-2-9b-it &nbsp;·&nbsp; LoRA r=32 &nbsp;·&nbsp; open-web-math &nbsp;·&nbsp; 120k samples &nbsp;·&nbsp; 3 epochs</div>

        {cards}
    </div>
    """
    display(HTML(html))

In [144]:
# ════════════════════════════════════════════════════════════════
# MAIN
# ════════════════════════════════════════════════════════════════

def run_eval():
    tokenizer, base_model, lora_model = load_models()
    results = []

    for i, stmt in enumerate(EVAL_STATEMENTS):
        print(f"[{i+1}/{len(EVAL_STATEMENTS)}] {stmt[:55]}...")
        base_out = generate(base_model, tokenizer, stmt)
        lora_out = generate(lora_model, tokenizer, stmt)
        results.append({"statement": stmt, "base": base_out, "lora": lora_out})

    render_html(results)
    return results

In [145]:
run_eval()

[LOAD] Tokenizer...
[LOAD] Base model...


Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

[LOAD] LoRA adapter...
[LOAD] Done.
[1/1] For any integer n, n² + n is even....


[{'statement': 'For any integer n, n² + n is even.',
  'base': 'Proof: We use induction on $n$. For our base case we show that when $n = 0$, then $n^2+n$ is even since $(0)^2+(0)=0=0\\cdot1$ where $\\frac{0}{2}=0.$ Now suppose for some arbitrary k ≥ 0 it’s true that $k^2+k$ is even so that there exists an element m such that $m \\in Z$ satisfying $k^2+k=2m$. Then consider what happens if you replace each instance of “k” by one more than itself to get ${(k+1)}^2+{k+1}$. Well notice how this can be rewritten as ${k}^2+2k+1)+({k}+1)$ which further simplifies into (${k}^2+k)+(2k+1)$. But now recall from above that "${k}^2+k$" equals "2m" hence all together we have "$2m +(2k+1) ="which after factoring out gives us $"2(m+k)+\\text{\\hspace{.3cm} }1"$ but because both "\\${m}\\$$ and \\$"\\textit{\\$k}"\\$are integers their sum must also be an integer say let\'s call it q therefore finally we obtain $\\$2q+\\text{\\hspace{.3cm}}1$\\$and thus showing that whenever $\\$$"${(\\textit{k}+\\text{\